# Tablero (Gradio) — Proyecto final

**Diplomado en Visualización de Datos y Creación de Tableros · Universidad del Valle**

| | |
|---|---|
| Estudiante | Jhon Fernando López Tamayo |
| Portafolio | https://github.com/jhonfernandolopez/actividad1-visualizacion-datos |
| Docente | Sebastián Barrios Caracas |

Tablero interactivo sobre los afiliados activos del Servicio de Salud Univalle (datos anonimizados),
construido con **Gradio** — la herramienta mostrada en la clase del módulo 5 (20-ago).

## Cómo correrlo
1. Arriba a la derecha: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → Python 3**.
2. Ejecutar todas las celdas (Entorno de ejecución → Ejecutar todas).
3. Al final va a aparecer un enlace público (termina en `.gradio.live`).

⚠️ **Ese enlace es temporal**: solo funciona mientras este notebook siga abierto y corriendo en Colab,
y expira solo a las **72 horas** aunque el notebook siga abierto. Si el profesor va a revisarlo después,
hay que volver a correr el notebook para generar un enlace nuevo.

In [ ]:
!pip install gradio -q

In [ ]:
# -*- coding: utf-8 -*-
"""Tablero del proyecto final (Diplomado Visualizacion de Datos - Univalle),
construido con Gradio, tal como lo enseno el profesor Sebastian en el modulo 5.
Corre en Google Colab: !pip install gradio -q, cambiar entorno a Python 3, ejecutar todo."""

import pandas as pd
import matplotlib.pyplot as plt
import gradio as gr

CSV_RAW = "https://raw.githubusercontent.com/jhonfernandolopez/actividad1-visualizacion-datos/main/clase-2-eda/afiliados_activos_anonimo.csv"

# --- paleta (coherente con el resto del proyecto) ---
AZUL = "#2a78d6"
NARANJA = "#eb6834"
GRIS_TEXTO = "#52514e"

df = pd.read_csv(CSV_RAW, encoding="utf-8")
bins = [0, 17, 30, 45, 60, 75, 120]
labels = ["0-17", "18-30", "31-45", "46-60", "61-75", "76+"]
df["grupo_edad"] = pd.cut(df["edad"], bins=bins, labels=labels, right=True, include_lowest=True)


def _estilo(ax, titulo):
    ax.set_title(titulo, fontsize=12, color="#0b0b0b", loc="left")
    ax.spines[["top", "right"]].set_visible(False)
    ax.tick_params(colors=GRIS_TEXTO, labelsize=9)
    for lbl in ax.get_xticklabels() + ax.get_yticklabels():
        lbl.set_color(GRIS_TEXTO)


def grafico_sexo():
    fig, ax = plt.subplots(figsize=(5, 3))
    conteo = df["sexo"].value_counts()
    colores = [AZUL if s == "F" else NARANJA for s in conteo.index]
    ax.barh(["Mujeres" if s == "F" else "Hombres" for s in conteo.index], conteo.values, color=colores)
    for i, v in enumerate(conteo.values):
        ax.text(v, i, f" {v:,}", va="center", fontsize=9, color="#0b0b0b")
    _estilo(ax, "Distribucion por sexo")
    fig.tight_layout()
    return fig


def grafico_edad():
    fig, ax = plt.subplots(figsize=(5, 3))
    conteo = df["grupo_edad"].value_counts().reindex(labels)
    ax.barh(labels, conteo.values, color=AZUL)
    for i, v in enumerate(conteo.values):
        ax.text(v, i, f" {v:,}", va="center", fontsize=9, color="#0b0b0b")
    _estilo(ax, "Grupo etario")
    fig.tight_layout()
    return fig


def grafico_estamento():
    fig, ax = plt.subplots(figsize=(6, 3.5))
    conteo = df["descripcion_estamento"].value_counts()
    top = conteo.head(7)
    otros = conteo.iloc[7:].sum()
    serie = pd.concat([top, pd.Series({"OTROS": otros})]).sort_values()
    ax.barh(serie.index, serie.values, color=AZUL)
    _estilo(ax, "Estamento (top 7 + otros)")
    fig.tight_layout()
    return fig


def grafico_parentesco():
    fig, ax = plt.subplots(figsize=(5, 3))
    conteo = df["parentesco"].fillna("COTIZANTE (titular)").value_counts().sort_values()
    ax.barh(conteo.index, conteo.values, color=AZUL)
    _estilo(ax, "Parentesco")
    fig.tight_layout()
    return fig


def grafico_etnia():
    fig, ax = plt.subplots(figsize=(5, 3))
    conteo = df["etnia"].value_counts().sort_values()
    ax.barh(conteo.index, conteo.values, color=AZUL)
    _estilo(ax, "Autorreconocimiento etnico")
    fig.tight_layout()
    return fig


def grafico_anio():
    fig, ax = plt.subplots(figsize=(10, 3.5))
    conteo = df["anio_afiliacion"].value_counts().sort_index()
    conteo = conteo[conteo.index >= 1990]
    ax.plot(conteo.index, conteo.values, color=AZUL, linewidth=2)
    ax.fill_between(conteo.index, conteo.values, color=AZUL, alpha=0.15)
    _estilo(ax, "Afiliaciones por ano (1990-2026)")
    fig.tight_layout()
    return fig


TOTAL = len(df)
PCT_MUJERES = round(100 * (df["sexo"] == "F").sum() / TOTAL, 1)
EDAD_PROM = round(df["edad"].mean(), 1)
ANTIGUEDAD_PROM = round(df["antiguedad_anios"].mean(), 1)

with gr.Blocks(title="Tablero - Afiliados Activos (AC)") as demo:
    gr.Markdown(
        f"""# Tablero — Afiliados Activos (AC), Servicio de Salud Univalle
**Proyecto final · Diplomado en Visualización de Datos y Creación de Tableros (Univalle)**

Datos anonimizados: documentos → hash, nombres → seudónimos, contacto → sintético.
N = **{TOTAL:,}** afiliados con estado AC (activo)."""
    )
    with gr.Row():
        gr.Textbox(f"{TOTAL:,}", label="Total afiliados (AC)", interactive=False)
        gr.Textbox(f"{PCT_MUJERES}%", label="% mujeres", interactive=False)
        gr.Textbox(f"{EDAD_PROM} años", label="Edad promedio", interactive=False)
        gr.Textbox(f"{ANTIGUEDAD_PROM} años", label="Antigüedad promedio", interactive=False)
    with gr.Row():
        gr.Plot(grafico_sexo)
        gr.Plot(grafico_edad)
    with gr.Row():
        gr.Plot(grafico_estamento)
        gr.Plot(grafico_parentesco)
    with gr.Row():
        gr.Plot(grafico_etnia)
    gr.Plot(grafico_anio)
    gr.Markdown(
        "*Fuente: SIIS Univalle, extracto anonimizado del análisis exploratorio (EDA). "
        "Generado para el proyecto final del diplomado.*"
    )

demo.launch(share=True, debug=False)